**Bienvenue dans ce Jupyter Notebook destiné à effectuer une analyse des sentiments à partir d'un dataset en Python**

In [1]:
!pip install nltk --break-system-packages
!pip install scikit-learn pandas joblib --break-system-packages
import sys
!{sys.executable} -m pip install pandas matplotlib numpy scikit-learn nltk --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
import re
import string

nltk.download('stopwords')  # Pour les mots vides
nltk.download('punkt_tab')  # Pour la tokenisation (déjà fait normalement)
nltk.download('wordnet')    # Pour la lemmatisation (notre prochaine étape !)

/home/joris/.local/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.9.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/joris/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
[nltk_data] Downloading package stopwords to /home/joris/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/joris/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/joris/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
df = pd.read_csv('IMDB_Dataset.csv')

In [4]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
df.shape

(50000, 2)

**Data Preprocessing**

In [6]:
mapping = {'positive': 1, 'negative': 0}

df['sentiment'] = df['sentiment'].map(mapping)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [7]:
df.value_counts

<bound method DataFrame.value_counts of                                                   review  sentiment
0      One of the other reviewers has mentioned that ...          1
1      A wonderful little production. <br /><br />The...          1
2      I thought this was a wonderful way to spend ti...          1
3      Basically there's a family where a little boy ...          0
4      Petter Mattei's "Love in the Time of Money" is...          1
...                                                  ...        ...
49995  I thought this movie did a down right good job...          1
49996  Bad plot, bad dialogue, bad acting, idiotic di...          0
49997  I am a Catholic taught in parochial elementary...          0
49998  I'm going to have to disagree with the previou...          0
49999  No one expects the Star Trek movies to be high...          0

[50000 rows x 2 columns]>

In [8]:
print(df['review'].value_counts())
print(df['sentiment'].value_counts())

review
Loved today's show!!! It was a variety and not solely cooking (which would have been great too). Very stimulating and captivating, always keeping the viewer peeking around the corner to see what was coming up next. She is as down to earth and as personable as you get, like one of us which made the show all the more enjoyable. Special guests, who are friends as well made for a nice surprise too. Loved the 'first' theme and that the audience was invited to play along too. I must admit I was shocked to see her come in under her time limits on a few things, but she did it and by golly I'll be writing those recipes down. Saving time in the kitchen means more time with family. Those who haven't tuned in yet, find out what channel and the time, I assure you that you won't be disappointed.                                                                                                                                                                                                         

In [9]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [10]:
df['review'] = df['review'].str.lower()

In [11]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. <br /><br />the...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1
...,...,...
49995,i thought this movie did a down right good job...,1
49996,"bad plot, bad dialogue, bad acting, idiotic di...",0
49997,i am a catholic taught in parochial elementary...,0
49998,i'm going to have to disagree with the previou...,0


In [12]:
df['review'] = df['review'].str.replace(r'<br\s*/?>', ' ', regex=True)

df['review'] = df['review'].str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)

In [13]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming t...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there s a family where a little boy ...,0
4,petter mattei s love in the time of money is...,1
...,...,...
49995,i thought this movie did a down right good job...,1
49996,bad plot bad dialogue bad acting idiotic di...,0
49997,i am a catholic taught in parochial elementary...,0
49998,i m going to have to disagree with the previou...,0


**Model Preparation and training**

In [14]:
df['tokens'] = df['review'].apply(word_tokenize)

In [15]:
df

,review,sentiment,tokens
0,one of the other reviewers has mentioned that ...,1,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production the filming t...,1,"[a, wonderful, little, production, the, filmin..."
2,i thought this was a wonderful way to spend ti...,1,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there s a family where a little boy ...,0,"[basically, there, s, a, family, where, a, lit..."
4,petter mattei s love in the time of money is...,1,"[petter, mattei, s, love, in, the, time, of, m..."
...,...,...,...
49995,i thought this movie did a down right good job...,1,"[i, thought, this, movie, did, a, down, right,..."
49996,bad plot bad dialogue bad acting idiotic di...,0,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,i am a catholic taught in parochial elementary...,0,"[i, am, a, catholic, taught, in, parochial, el..."
49998,i m going to have to disagree with the previou...,0,"[i, m, going, to, have, to, disagree, with, th..."


In [16]:
stop_words = set(stopwords.words('english'))
df['tokens'] = df['tokens'].apply(lambda x : [word for word in x if word not in stop_words])

In [17]:
df

,review,sentiment,tokens
0,one of the other reviewers has mentioned that ...,1,"[one, reviewers, mentioned, watching, 1, oz, e..."
1,a wonderful little production the filming t...,1,"[wonderful, little, production, filming, techn..."
2,i thought this was a wonderful way to spend ti...,1,"[thought, wonderful, way, spend, time, hot, su..."
3,basically there s a family where a little boy ...,0,"[basically, family, little, boy, jake, thinks,..."
4,petter mattei s love in the time of money is...,1,"[petter, mattei, love, time, money, visually, ..."
...,...,...,...
49995,i thought this movie did a down right good job...,1,"[thought, movie, right, good, job, creative, o..."
49996,bad plot bad dialogue bad acting idiotic di...,0,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,i am a catholic taught in parochial elementary...,0,"[catholic, taught, parochial, elementary, scho..."
49998,i m going to have to disagree with the previou...,0,"[going, disagree, previous, comment, side, mal..."


In [18]:
lemmatizer = WordNetLemmatizer()
df['tokens'] = df['tokens'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

In [19]:
df['cleaned_content'] = df['tokens'].apply(lambda x: ' '.join(x))

In [20]:
df

,review,sentiment,tokens,cleaned_content
0,one of the other reviewers has mentioned that ...,1,"[one, reviewer, mentioned, watching, 1, oz, ep...",one reviewer mentioned watching 1 oz episode h...
1,a wonderful little production the filming t...,1,"[wonderful, little, production, filming, techn...",wonderful little production filming technique ...
2,i thought this was a wonderful way to spend ti...,1,"[thought, wonderful, way, spend, time, hot, su...",thought wonderful way spend time hot summer we...
3,basically there s a family where a little boy ...,0,"[basically, family, little, boy, jake, think, ...",basically family little boy jake think zombie ...
4,petter mattei s love in the time of money is...,1,"[petter, mattei, love, time, money, visually, ...",petter mattei love time money visually stunnin...
...,...,...,...,...
49995,i thought this movie did a down right good job...,1,"[thought, movie, right, good, job, creative, o...",thought movie right good job creative original...
49996,bad plot bad dialogue bad acting idiotic di...,0,"[bad, plot, bad, dialogue, bad, acting, idioti...",bad plot bad dialogue bad acting idiotic direc...
49997,i am a catholic taught in parochial elementary...,0,"[catholic, taught, parochial, elementary, scho...",catholic taught parochial elementary school nu...
49998,i m going to have to disagree with the previou...,0,"[going, disagree, previous, comment, side, mal...",going disagree previous comment side maltin on...


In [21]:
X = df['cleaned_content']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [22]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [23]:
print(X_train_tfidf)

  (0, 33269)	0.08073432328927974
  (0, 76391)	0.040834115741908206
  (0, 48)	0.037495350214899394
  (0, 8397)	0.07995458147051107
  (0, 64633)	0.05185509167800472
  (0, 78736)	0.06917205403294709
  (0, 58959)	0.07145820254858934
  (0, 21433)	0.10821558501607108
  (0, 48072)	0.07425783005830162
  (0, 1896)	0.04955780247201142
  (0, 58409)	0.07309352382082258
  (0, 26213)	0.06880038235128134
  (0, 51606)	0.08023253204719326
  (0, 22358)	0.08378389070178857
  (0, 81913)	0.0673147256391525
  (0, 37116)	0.0664218323102826
  (0, 18770)	0.04657959951727415
  (0, 3191)	0.05038275600409468
  (0, 56752)	0.03217076613907953
  (0, 30767)	0.03506414259768321
  (0, 1902)	0.03734638428011066
  (0, 38548)	0.05838542014842136
  (0, 11270)	0.07735758169765514
  (0, 35398)	0.05945922525535389
  (0, 72928)	0.08731993462487558
  :	:
  (39999, 60347)	0.07085474449236132
  (39999, 45402)	0.05074593982724316
  (39999, 67019)	0.09026094429302929
  (39999, 65674)	0.08714659919878415
  (39999, 45329)	0.086356443

In [24]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [25]:
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87      4961
           1       0.88      0.85      0.86      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



**Testons notre modèle !**

In [26]:
new_data = ["I didn't like this movie ! "]

new_data = [comment.lower() for comment in new_data]

new_data = [comment.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation))) for comment in new_data]

new_data = [re.sub(r'\s+', ' ', comment.strip()) for comment in new_data]

new_data_tfidf = vectorizer.transform(new_data)

prediction = model.predict(new_data_tfidf)
prediction_proba = model.predict_proba(new_data_tfidf)

print("Prediction : ", prediction[0])
print("Probabilities :", prediction_proba)



Prediction :  0
Probabilities : [[0.50360005 0.49639995]]


In [27]:
import joblib

joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']